In [ ]:
# Import des données traitées
from numpy import select
import pandas as pd
import duckdb
pd.set_option('display.max_columns', None)

chemin_fichier_order = r"F_orders.parquet"
chemin_fichier_product = r"D_products.parquet"
chemin_fichier_canal = r"d_facture_clean.parquet"

df_ML_customer=duckdb.query(f"""
    with order_complete as(
        SELECT
            o.*,
            CASE
                WHEN o.TotalDiscountGBP=0 THEN 1
                ELSE 0
            END AS check_promotion,
            p.ProductCategory_Propre,
            c.SalesChannel,
        From read_parquet('{chemin_fichier_order}') as o
        LEFT JOIN read_parquet('{chemin_fichier_product}') as p
            ON p.StockCode = o.StockCode
        LEFT JOIN read_parquet('{chemin_fichier_canal}') as c
            ON c.Invoice=o.Invoice
    )                        

    select 
        CustomerID_clean,
        SUM(TotalPriceGBP) as total_CA,
        SUM(TotalPriceGBP_WithDiscount) as total_CA_apres_remise,
        SUM(Quantity_clean) as Total_quantite,
        SUM(TotalPriceGBP)/SUM(Quantity_clean) as panier_moyen,
        SUM(TotalDiscountGBP)/SUM(TotalPriceGBP) as remise_moyenne,
        SUM(check_promotion) as nb_achat_avec_promo,
        SUM(check_promotion)/SUM(DISTINCT(OrderLineID)) as pourcent_achat_avec_promo
    From order_complete
    GROUP BY CustomerID_clean
""").df()

In [3]:
def explorer_dataset(dataset):
        print(f"Start {'-'*10} Pour ce dataset, la taille est de {dataset.shape} {'-'*10}")
        print('-'*30)
        display(dataset.head(3))
        print('-'*30)
        display(dataset.tail(3))
        print('-'*30)
        print(f"Présences de valeurs 'NaN'.")
        print(f"{dataset.isna().sum()}")
        display(dataset[dataset.isna().any(axis=1)])
        print('-'*30)
        print(f"Les doublons")
        print(f"{dataset.duplicated().sum()}")
        print('-'*30)
        print(f"Les types de données")
        print(f"{dataset.dtypes}")
        print('-'*30)
        print(f"Description")
        display(dataset.describe().T)
        print("FIN"+'-'*100)

In [8]:
explorer_dataset(df_ML_customer)

Start ---------- Pour ce dataset, la taille est de (5943, 3) ----------
------------------------------


,CustomerID_clean,total_CA,Total_quantite
0,12380.0,35460.840096,3353.0
1,13198.0,35041.060028,3389.0
2,13821.0,5730.850025,486.0


------------------------------


,CustomerID_clean,total_CA,Total_quantite
5940,17129.0,154.939995,5.0
5941,17310.0,4249.140015,402.0
5942,17641.0,17.820000,1.0


------------------------------
Présences de valeurs 'NaN'.
CustomerID_clean    0
total_CA            0
Total_quantite      0
dtype: int64


,CustomerID_clean,total_CA,Total_quantite


------------------------------
Les doublons
0
------------------------------
Les types de données
CustomerID_clean        str
total_CA            float64
Total_quantite      float64
dtype: object
------------------------------
Description


,count,mean,std,min,25%,50%,75%,max
total_CA,5943.0,24776.250538,258750.704711,1.01,2150.509975,5755.819992,16461.41515,1.822550e+07
Total_quantite,5943.0,2089.351338,23421.199661,1.00,184.000000,483.000000,1370.00000,1.670108e+06


FIN----------------------------------------------------------------------------------------------------
